# LanceDB 

TO DO : 
- check embeddings shape in model to be able to set dimensions right 


In [1]:
import lancedb 
import pandas as pd 
# creating an empty folder 
db = lancedb.connect(uri="vector_db")

db

LanceDBConnection(uri='/Users/mirandalyng/Desktop/Data_engineer/GitHub_repos/AI_ops_course/09_lancedb/vector_db')

In [2]:
db.uri

'/Users/mirandalyng/Desktop/Data_engineer/GitHub_repos/AI_ops_course/09_lancedb/vector_db'

### Create a table 

In [3]:
import json 

with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

-  Mode = overwrite

In [4]:
table_animals = db.create_table("animals_text", exist_ok=True, data = data) 

table_animals

LanceTable(name='animals_text', _conn=LanceDBConnection(uri='/Users/mirandalyng/Desktop/Data_engineer/GitHub_repos/AI_ops_course/09_lancedb/vector_db'))

In [5]:
# it's a pyarrow table
table_animals.head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [6]:
# converting to pandas
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


### Add more data

In [7]:
more_data = [
    {"text": "A panda eating bamboo peacefully.",
        "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

table_animals.add(more_data)

AddResult(version=5)

In [8]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


### Create an empty table and then delete it

In [9]:
# LanceModel is a Pydantic model that can be converted to LanceDB table
from lancedb.pydantic import LanceModel 

class JokesSchema(LanceModel): 
    joke: str 
    rating: int 

db.create_table(name = "jokes", schema=JokesSchema, exist_ok=True)

db

ValueError: Schema Error: Provided schema does not match existing table schema

In [ ]:
db.table_names()

/var/folders/49/f5021v_n7cb6ghg2_c0drpn00000gn/T/ipykernel_7381/763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text', 'jokes']

In [10]:
db.drop_table("jokes")

In [11]:
db.table_names()

/var/folders/49/f5021v_n7cb6ghg2_c0drpn00000gn/T/ipykernel_7973/763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text']

### Open existing table

In [12]:
db.open_table("animals_text").head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

### Vecor search in lanceDB
Uses ANN - approximate nearest neighbours for searching vectors

- search with vector directly
- note that the vector column is already an example embedding simulated with LLM

A more realistic approach is to just have text records, use an existing embedding model and embedd those text records, then embed the query vector and find closest match

In [13]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [14]:
query_vector = [0.5, 0.2, 0.9]

table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354


### Embedding API 
- idea: want to put in text -> and it eill automatically generate vector embeddings
- put in a query and it will automaticcalyy generate vector embeddings 
- calculate closest distances 


LanceDB provides an Embeddings API that allows you to easily generate and store vector embeddings for your data. This is particularly useful for applications such as semantic search, recommendation systems, and clustering.

We'll use embeddings model from sentence-transformer here


In [24]:
from lancedb.pydantic import LanceModel, Vector 
from lancedb.embeddings import get_registry

model = get_registry().get("sentence-transformers").create(name="intfloat/multilingual-e5-small")

class JokeModel(LanceModel):
    joke: str = model.SourceField() #marks tezt as input to embedding function
    vector: Vector(model.ndims()) = model.VectorField() #vector is destination of computed embedding




Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9816.60it/s]


In [25]:

table_jokes = db.create_table("jokes", schema=JokeModel,mode="overwrite")

table_jokes

[2026-11-09T11:23:09Z WARN  lance::dataset::write::insert] No existing dataset at /Users/mirandalyng/Desktop/Data_engineer/GitHub_repos/AI_ops_course/09_lancedb/vector_db/jokes.lance, it will be created


LanceTable(name='jokes', _conn=LanceDBConnection(uri='/Users/mirandalyng/Desktop/Data_engineer/GitHub_repos/AI_ops_course/09_lancedb/vector_db'))

In [26]:
with open("data/jokes.json", "r") as file: 
    jokes_data = json.loads(file.read())

jokes_data

{'jokes': ['Parallel lines have so much in common—it’s sad they’ll never meet.',
  'ETL stands for “Extract, Transform, Leave for the next person.”',
  'What do you call a snake that runs your scripts? A py-thon (ton of errors).',
  'Gold walks into a bar. The bartender says, “Au, get out!”',
  'C# devs don’t argue; they just throw exceptions.',
  'I asked the data lake if it had my file. It said: “Sure… somewhere.”',
  'My math teacher said I’m average… how mean!',
  'Python is so friendly even whitespace gets a say.',
  'The C# compiler walked into a bar. The bartender said, “You’re not allowed here!” It replied, “But I’ve got excellent syntax!”',
  'Never trust an atom—they make up everything.',
  'Why was the equal sign so humble? It knew it wasn’t < or > than anyone.',
  'My Python code works on the first try… said no one ever.',
  'Why did the chemist ground his kids? Because they were being basic.',
  "Our data pipeline broke… but don't worry, it was by design. We call it a feat

In [27]:
df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)

df_jokes.head()


,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [28]:
df_jokes["joke"].iloc[0]

'Parallel lines have so much in common—it’s sad they’ll never meet.'

##### add data to table

In [29]:
table_jokes.add(df_jokes)

AddResult(version=2)

In [30]:
table_jokes.head()

pyarrow.Table
joke: string not null
vector: fixed_size_list<item: float>[384]
  child 0, item: float
----
joke: [["Parallel lines have so much in common—it’s sad they’ll never meet.","ETL stands for “Extract, Transform, Leave for the next person.”","What do you call a snake that runs your scripts? A py-thon (ton of errors).","Gold walks into a bar. The bartender says, “Au, get out!”","C# devs don’t argue; they just throw exceptions."]]
vector: [[[0.037198003,-0.019917808,-0.051840983,-0.032692898,0.10485611,...,-0.07006689,0.04333398,0.04714111,0.06722881,0.05570244],[0.05049742,-0.0064256676,-0.07101135,-0.02772608,0.044613183,...,-0.015193862,-0.0073572197,0.032429952,0.027537309,0.07036025],[0.07316833,-0.004550006,-0.027777126,-0.05414758,0.04447575,...,-0.05463064,0.041998446,0.056127965,0.05367643,0.0453344],[0.045366444,-0.052969992,-0.06690077,-0.028929906,0.1135727,...,-0.060623806,0.08459842,0.050644685,0.06691328,0.011901228],[0.028011428,0.0046460265,-0.05766367,-0.06480118

In [31]:
table_jokes.to_pandas()

,joke,vector
0,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0...."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.050497, -0.006426, -0.071011, -0.027726, 0...."
2,What do you call a snake that runs your script...,"[0.073168, -0.00455, -0.027777, -0.054148, 0.0..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.045366, -0.05297, -0.066901, -0.02893, 0.11..."
4,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0..."
5,I asked the data lake if it had my file. It sa...,"[0.069376, -0.0138, -0.052487, -0.050929, 0.08..."
6,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0...."
7,Python is so friendly even whitespace gets a say.,"[0.020338, 0.010201, -0.026083, -0.032346, 0.0..."
8,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1..."
9,Never trust an atom—they make up everything.,"[0.059282, -0.023624, -0.089997, -0.071139, 0...."


In [32]:
table_jokes.to_pandas()["vector"][0]

array([ 3.71980034e-02, -1.99178085e-02, -5.18409833e-02, -3.26928981e-02,
        1.04856111e-01, -7.87372701e-03,  3.17407809e-02, -7.14515569e-03,
        9.07414407e-02,  3.75400484e-02,  2.76287347e-02,  1.52677540e-02,
        2.58133914e-02, -4.01636809e-02, -6.00590557e-02,  4.96724062e-02,
        5.30899987e-02, -5.45793325e-02, -1.04770891e-01, -5.57886101e-02,
        2.83987820e-02,  3.01512238e-03, -4.67016250e-02,  7.09141418e-02,
        3.86798754e-02,  7.03528151e-02, -3.87883186e-02,  6.77997619e-02,
        2.99662966e-02, -5.36424592e-02, -2.42412798e-02, -6.57098442e-02,
        3.89403105e-02, -2.38893833e-02,  2.19925120e-02,  2.89189145e-02,
       -2.07227599e-02, -5.95963858e-02,  3.77600342e-02, -6.66520149e-02,
       -3.60499695e-02,  5.38722351e-02,  4.61162999e-02,  6.78815469e-02,
        7.88854584e-02,  2.33219732e-02, -3.27598788e-02,  3.68100107e-02,
       -5.52664511e-02, -8.71163979e-03, -4.17464338e-02,  5.86064942e-02,
        3.62174325e-02,  

### Perform searching 

In [34]:
table_jokes.search("data engineering jokes").limit(8).to_pandas()

,joke,vector,_distance
0,Why do data engineers hate nature? Too many un...,"[0.057509, 0.012957, -0.081348, -0.042842, 0.0...",0.263454
1,"Data engineer motto: If it works, don’t touch ...","[0.003134, -0.036774, -0.050577, -0.060756, 0....",0.275054
2,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.309797
3,"Our data pipeline broke… but don't worry, it w...","[0.032507, -0.009452, -0.025917, -0.060267, 0....",0.314785
4,I asked the data lake if it had my file. It sa...,"[0.069376, -0.0138, -0.052487, -0.050929, 0.08...",0.330099
5,I tried to explain async/await to my friend… n...,"[-0.004627, -0.014231, -0.044107, -0.028893, 0...",0.332070
6,C# programmers love coffee—they're always work...,"[-0.011444, -0.021634, -0.055446, -0.035535, 0...",0.340251
7,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.340883


In [35]:
table_jokes.search("give me jokes on C#").limit(8).to_pandas()

,joke,vector,_distance
0,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.234623
1,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0...",0.260993
2,Why is C# like a musical? It has so many classes.,"[0.021061, -0.003477, -0.029677, -0.060304, 0....",0.263107
3,C# programmers love coffee—they're always work...,"[-0.011444, -0.021634, -0.055446, -0.035535, 0...",0.278925
4,Why did the C# developer go broke? He kept usi...,"[0.020793, 0.007793, -0.040906, -0.02401, 0.03...",0.296744
5,I tried to explain async/await to my friend… n...,"[-0.004627, -0.014231, -0.044107, -0.028893, 0...",0.310200
6,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.323905
7,Why did the Python programmer get bitten? Beca...,"[0.029594, 0.007605, -0.018481, -0.004575, 0.0...",0.337853


In [36]:
table_jokes.search("give me chemistry related jokes").limit(8).to_pandas()

,joke,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.224214
1,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.246833
2,Chemists have all the solutions… mostly in bea...,"[0.025907, -0.022183, -0.06204, -0.036586, 0.0...",0.283978
3,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.293797
4,Never trust an atom—they make up everything.,"[0.059282, -0.023624, -0.089997, -0.071139, 0....",0.308730
5,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.320930
6,"Dear algebra, please stop asking us to find yo...","[0.05175, 0.00825, -0.020384, -0.038882, 0.084...",0.326247
7,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.327856


### Hybrid search 
hybrid search combines traditional keyword-based search with vector-based search to provide more relevant results. This approach leverages the strengths of both methods, allowing for more accurate and context-aware retrieval of information.

In [ ]:
# to enable keyword based search on joke column
table_jokes.create_fts_index("joke", replace=True)

In [37]:
from lancedb import rerankers

# reranks the results from a vector and full text search

reranker = rerankers.RRFReranker()

result = table_jokes.search(
    "give me nature related jokes", 
    query_type="hybrid", 
    vector_column_name="vector", 
    fts_columns="joke", 

    ).rerank(reranker).limit(8).to_pandas()
# somewhat different to before
result

,joke,vector,_relevance_score
0,Why do data engineers hate nature? Too many un...,"[0.057509, 0.012957, -0.081348, -0.042842, 0.0...",0.032522
1,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.016393
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.015873
3,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.015625
4,I asked the data lake if it had my file. It sa...,"[0.069376, -0.0138, -0.052487, -0.050929, 0.08...",0.015385
5,I tried to explain async/await to my friend… n...,"[-0.004627, -0.014231, -0.044107, -0.028893, 0...",0.015152
6,"My pipeline failed again, so I’m logging out—l...","[0.014553, -0.021881, 0.002843, -0.057108, 0.0...",0.014925
7,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.014706


### Rule of thumb
- For exact matching -> FTS (Full text search)
- For meaning based matching -> Vector search
- Unpredictable or mixed queries -> Hybrid search